# sudregex Tutorial Notebook

**Version:** 0.1.7

This notebook walks through the core `sudregex` workflows from start to finish:

1. Install and import the package
2. Load note data into a pandas DataFrame
3. Use the built-in pattern library and termslist
4. Run `extract_df()` in memory
5. Inspect results and match previews
6. Run `validate_pattern_library()` against labeled examples
7. Compare serial vs parallel backends
8. Run the file-based CLI for large datasets
9. Handle headerless text files with custom separators
10. Bring your own pattern library

---

> **Note:** `sudregex` is a regex-based NLP extraction engine for clinical notes.
> It is designed for substance use disorder (SUD) phenotyping from EHR text,
> but the pattern library framework applies to any clinical extraction task.

## 1. Installation

Install from PyPI:

```bash
pip install sudregex
```

Or install from source in editable mode (for development):

```bash
git clone https://github.com/quantitativenurse/sud-regex.git
cd sud-regex
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -U pip
pip install -e .[dev]
```

**Supported Python versions:** 3.9, 3.10, 3.11, 3.12, 3.13

In [4]:
import pandas as pd
import sudregex as sud

print("sudregex version:", sud.__version__)

sudregex version: 0.1.7


## 2. Load note data

Your DataFrame needs at minimum:
- A **note identifier** column (unique per note)
- A **note text** column

A **person identifier** column is optional but recommended — it gets reattached
to the output so you can link results back to patients.

Below we create a small in-memory example with 4 synthetic clinical notes.
These cover common clinical scenarios: active use, negated use,
treatment engagement, and a discharge context (which should be excluded).

In [5]:
df = pd.DataFrame(
    {
        "patient_id": ["P001", "P002", "P003", "P004"],
        "note_id":    ["1001", "1002", "1003", "1004"],
        "note_text": [
            # Active use — should fire on illicit drug and opioid items
            "Patient reports daily heroin use for the past 2 years. "
            "He admits to obtaining oxycodone from multiple providers.",

            # Negated use — 'denies' and 'no' should suppress these hits
            "Denies opioid use. No heroin, fentanyl, or oxycodone use reported.",

            # Treatment engagement — buprenorphine started for OUD
            "Started buprenorphine-naloxone for opioid use disorder. "
            "Patient is engaged and attending NA meetings weekly.",

            # Discharge context — should be excluded by discharge gate
            "Discharge instructions: take oxycodone only as prescribed for pain.",
        ],
    }
)

df

,patient_id,note_id,note_text
0,P001,1001,Patient reports daily heroin use for the past ...
1,P002,1002,"Denies opioid use. No heroin, fentanyl, or oxy..."
2,P003,1003,Started buprenorphine-naloxone for opioid use ...
3,P004,1004,Discharge instructions: take oxycodone only as...


## 3. Use the built-in pattern library and termslist

`sudregex` ships with:
- `sud.pattern_library_abc` — the ABC OUD checklist (20 items covering opioid misuse behaviors)
- `sud.default_termslist` — a grouped vocabulary dict with `opioid_terms`, `alcohol_terms`, and `chronic_pain_terms`

You pass a specific term group to `terms_active` to tell the substance gate
which vocabulary to require near each match.

In [6]:
# Access the built-in pattern library and termslist
pattern_library = sud.pattern_library_abc
termslist = sud.default_termslist

# Inspect what's available
print("Pattern library type:", type(pattern_library))
print("Pattern library items:", list(pattern_library.keys()))
print()
print("Termslist groups:", list(termslist.keys()))
print("Number of opioid terms:", len(termslist["opioid_terms"]))

Pattern library type: <class 'dict'>
Pattern library items: ['1a', '1b', '1c', '2', '3', '4', '5a', '5b', '6', '7a', '7b', '8', '9', '10', '11a', '11b', '12a', '12b', '13', '14', '15', '16', '17', '18a', '18b', '19', '20']

Termslist groups: ['alcohol_terms', 'opioid_terms', 'chronic_pain_terms']
Number of opioid terms: 133


In [7]:
# Inspect a single pattern library item to understand the structure
# Each item has: lab (label), pat (regex), col_name, opioid/substance, negation, preview, common_fp
import json
item = pattern_library["1a"]
print("Item 1a:")
for k, v in item.items():
    if k != "pat":  # skip printing the full regex for readability
        print(f"  {k}: {v}")
print(f"  pat: {str(item['pat'])[:80]}...")

Item 1a:
  lab: Since last visit: #1 "Patient used illicit drugs or evidences problem drinking" #1a Illicit drugs
  col_name: illicit_drug_use
  opioid: True
  negation: True
  preview: False
  pat: ((illicit drug)|mariju|cocai|heroin|polysubst|methamphetamine|amphetamine|\becst...


## 4. Run `extract_df()` — in-memory extraction

`extract_df()` is the primary API for notebook and development workflows.
It takes a DataFrame and returns a result DataFrame with one row per note
and one column per pattern library item.

**Key parameters:**
- `pattern_library` — your checklist dict
- `termslist` + `terms_active` — substance vocabulary for the substance gate
- `remove_linebreaks` — normalizes note text (recommended: `True`)
- `exclude_discharge_mentions` — drops hits near discharge instructions
- `negation_scope` — `"left"` (default), `"right"`, or `"both"`
- `return_previews_df` — returns a second DataFrame with match context snippets

In [8]:
import time

start = time.time()

result_df, previews_df = sud.extract_df(
    df=df,
    pattern_library=pattern_library,
    termslist=termslist,
    terms_active="opioid_terms",   # use the opioid term group for substance gating
    person_column="patient_id",    # reattached to output for patient-level linkage
    id_column="note_id",
    include_note_text=True,        # keep note text in output for QA
    remove_linebreaks=True,        # normalize whitespace and break markers
    exclude_discharge_mentions=True,  # suppress hits inside discharge instructions
    negation_scope="left",         # look for negation cues to the LEFT of each match
    preview_count=5,               # emit up to 5 preview snippets per item
    preview_span=120,              # chars of context around each match
    parallel=False,                # serial mode for development
    debug=False,
    return_previews_df=True,       # return previews as a second DataFrame
)

elapsed = round(time.time() - start, 2)
print(f"Extraction complete in {elapsed}s")
print(f"result_df shape:   {result_df.shape}")
print(f"previews_df shape: {previews_df.shape}")

Extraction complete in 0.21s
result_df shape:   (4, 72)
previews_df shape: (0, 6)


## 5. Inspect results

The result DataFrame has:
- `patient_id`, `note_id` — identifier columns (always first)
- One column per pattern library item (base match count)
- `_SUBSTANCE_MATCHED` columns — matches that also had a substance term nearby
- `_NEG` columns — matches that survived the negation gate

**Column values are match counts, not binary flags.**
A value of `2` means the pattern matched twice in that note.

In [10]:
# Show all columns
print("Output columns:")
for col in result_df.columns:
    print(" ", col)

Output columns:
  patient_id
  note_id
  illicit_drug_use
  illicit_drug_use_SUBSTANCE_MATCHED
  illicit_drug_use_SUBSTANCE_MATCHED_NEG
  problematic_alcohol_use
  problematic_alcohol_use_NEG
  dui_history
  dui_history_NEG
  medication_hoarding
  medication_hoarding_SUBSTANCE_MATCHED
  medication_hoarding_SUBSTANCE_MATCHED_NEG
  excess_narcotic_use
  excess_narcotic_use_SUBSTANCE_MATCHED
  excess_narcotic_use_SUBSTANCE_MATCHED_NEG
  ran_out_of_meds_early
  ran_out_of_meds_early_SUBSTANCE_MATCHED
  ran_out_of_meds_early_SUBSTANCE_MATCHED_NEG
  increased_opioid_use
  increased_opioid_use_SUBSTANCE_MATCHED
  increased_opioid_use_SUBSTANCE_MATCHED_NEG
  dose_escalation
  dose_escalation_SUBSTANCE_MATCHED
  dose_escalation_SUBSTANCE_MATCHED_NEG
  nonadherence_prn
  nonadherence_prn_SUBSTANCE_MATCHED
  multiple_providers_primary
  multiple_providers_primary_SUBSTANCE_MATCHED
  multiple_providers_primary_SUBSTANCE_MATCHED_NEG
  multiple_providers_alt
  multiple_providers_alt_SUBSTANCE_MATCHE

In [11]:
# Show results for a few key items
# Item 1a = illicit drug use, item 7a = multiple providers, item 16 = opioid mention
key_cols = ["patient_id", "note_id",
            "illicit_drug_use", "illicit_drug_use_SUBSTANCE_MATCHED", "illicit_drug_use_SUBSTANCE_MATCHED_NEG",
            "multiple_providers_primary", "multiple_providers_primary_SUBSTANCE_MATCHED_NEG",
            "opioid_mention", "opioid_mention_SUBSTANCE_MATCHED_NEG"]

# Only show columns that exist in output
available = [c for c in key_cols if c in result_df.columns]
result_df[available]

,patient_id,note_id,illicit_drug_use,illicit_drug_use_SUBSTANCE_MATCHED,illicit_drug_use_SUBSTANCE_MATCHED_NEG,multiple_providers_primary,multiple_providers_primary_SUBSTANCE_MATCHED_NEG,opioid_mention,opioid_mention_SUBSTANCE_MATCHED_NEG
0,P001,1001,1,1,1,1,1,1,1
1,P002,1002,1,1,0,0,0,3,0
2,P003,1003,0,0,0,0,0,2,2
3,P004,1004,0,0,0,0,0,1,0


In [12]:
# Quick summary: which items fired on at least one note?
# Exclude identifier and text columns
signal_cols = [c for c in result_df.columns
               if c not in ["patient_id", "note_id", "note_text"]]

fired = (result_df[signal_cols] > 0).sum().sort_values(ascending=False)
print("Items that fired (notes with at least 1 match):")
print(fired[fired > 0].to_string())

Items that fired (notes with at least 1 match):
opioid_mention_SUBSTANCE_MATCHED                    4
opioid_mention                                      4
opioid_mention_SUBSTANCE_MATCHED_NEG                2
illicit_drug_use_SUBSTANCE_MATCHED                  2
illicit_drug_use                                    2
multiple_providers_primary                          1
multiple_providers_primary_SUBSTANCE_MATCHED_NEG    1
illicit_drug_use_SUBSTANCE_MATCHED_NEG              1
multiple_providers_primary_SUBSTANCE_MATCHED        1
multiple_providers_alt                              1
multiple_providers_alt_SUBSTANCE_MATCHED            1
multiple_providers_alt_SUBSTANCE_MATCHED_NEG        1


## 6. Inspect match previews

When `return_previews_df=True` and `preview_count > 0`, sudregex returns
a `previews_df` with context snippets around each match.

This is useful for:
- Verifying that the pattern is matching what you expect
- Identifying false positives and false negatives
- Debugging negation and substance gate behavior

The `previews` feature only emits snippets for items where `preview=True`
in the pattern library definition.

In [13]:
# Check which pattern library items have preview=True
preview_items = [k for k, v in pattern_library.items() if v.get("preview")]
print("Items with preview=True:", preview_items)
print()
print("previews_df shape:", previews_df.shape)
previews_df

Items with preview=True: []

previews_df shape: (0, 6)


,item_key,note_id,span_start,span_end,snippet,snippet_marked


## 7. Validate against labeled examples

`validate_pattern_library()` measures per-item precision, recall, and F1
against a set of human-labeled examples.

**Input format:** a DataFrame (or text file) with three columns:
- `item_key` — the pattern library key (e.g. `"1a"`, `"16"`)
- `expected` — ground truth label: `1` (present) or `0` (absent)
- `note_text` — the note text

**Returns:**
- `detailed` — row-level results with actual match, mismatch flag, failure reason
- `by_item` — per-item precision, recall, F1, TP, FP, FN

In [14]:
from sudregex.validation import validate_pattern_library

# Build a small labeled example set
# item_key | expected | note_text
val_df = pd.DataFrame([
    # Item 1a — illicit drug use
    {"item_key": "1a", "expected": 1,
     "note_text": "Patient reports daily heroin use for the past 2 years. "
                  "He admits to obtaining oxycodone from multiple providers."},
    {"item_key": "1a", "expected": 0,
     "note_text": "Denies opioid use. No heroin, fentanyl, or oxycodone use reported."},
    {"item_key": "1a", "expected": 0,
     "note_text": "Discharge instructions: take oxycodone only as prescribed."},

    # Item 7a — multiple providers
    {"item_key": "7a", "expected": 1,
     "note_text": "Patient admits to obtaining oxycodone from multiple providers "
                  "over the past month."},
    {"item_key": "7a", "expected": 0,
     "note_text": "She is seen by multiple specialists for her chronic conditions."},

    # Item 16 — opioid mention
    {"item_key": "16", "expected": 1,
     "note_text": "Started buprenorphine-naloxone for opioid use disorder."},
    {"item_key": "16", "expected": 0,
     "note_text": "Patient has no pain complaints today."},
])

print("Validation set shape:", val_df.shape)
val_df

Validation set shape: (7, 3)


,item_key,expected,note_text
0,1a,1,Patient reports daily heroin use for the past ...
1,1a,0,"Denies opioid use. No heroin, fentanyl, or oxy..."
2,1a,0,Discharge instructions: take oxycodone only as...
3,7a,1,Patient admits to obtaining oxycodone from mul...
4,7a,0,She is seen by multiple specialists for her ch...
5,16,1,Started buprenorphine-naloxone for opioid use ...
6,16,0,Patient has no pain complaints today.


In [15]:
# Run validation
# substance_terms filters to only fire when a substance term is nearby
detailed, by_item = validate_pattern_library(
    pattern_library=pattern_library,
    examples=val_df,
    substance_terms=termslist["opioid_terms"],
)

print("Validation complete.")
print()
print("Per-item summary:")
print(by_item.to_string(index=False))

Validation complete.

Per-item summary:
item_key  n  tp  fp  fn  precision  recall  f1
      16  2   1   0   0        1.0     1.0 1.0
      1a  3   1   0   0        1.0     1.0 1.0
      7a  2   1   0   0        1.0     1.0 1.0
   TOTAL  7   3   0   0        1.0     1.0 1.0


In [16]:
# The detailed output shows exactly what happened for each example
# failure_reason tells you WHY a match was suppressed:
#   negated        — negation cue found to the left
#   needs_substance — no substance term found nearby
#   common_fp       — a known false-positive term was found nearby
#   no_raw_hit      — the pattern did not match at all
detailed[["item_key", "expected", "actual_match", "mismatch", "failure_reason"]]

,item_key,expected,actual_match,mismatch,failure_reason
0,1a,1,1,0,
1,1a,0,0,0,negated
2,1a,0,0,0,no_raw_hit
3,7a,1,1,0,
4,7a,0,0,0,no_raw_hit
5,16,1,1,0,
6,16,0,0,0,no_raw_hit


In [9]:
result_df.head()

,patient_id,note_id,illicit_drug_use,illicit_drug_use_SUBSTANCE_MATCHED,illicit_drug_use_SUBSTANCE_MATCHED_NEG,problematic_alcohol_use,problematic_alcohol_use_NEG,dui_history,dui_history_NEG,medication_hoarding,...,minimal_relief,minimal_relief_SUBSTANCE_MATCHED,opioid_tolerance,opioid_tolerance_SUBSTANCE_MATCHED,opioid_tolerance_SUBSTANCE_MATCHED_NEG,med_agreement_difficulty,significant_other_concern,significant_other_concern_SUBSTANCE_MATCHED,significant_other_concern_SUBSTANCE_MATCHED_NEG,note_text
0,P001,1001,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Patient reports daily heroin use for the past ...
1,P002,1002,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"Denies opioid use. No heroin, fentanyl, or oxy..."
2,P003,1003,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Started buprenorphine-naloxone for opioid use ...
3,P004,1004,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Discharge instructions: take oxycodone only as...


## 8. Serial vs parallel backends

For large datasets, `sudregex` supports two parallel backends:
- `pandarallel` — multiprocessing via pandas `.parallel_apply()`
- `loky` — joblib-based multiprocessing, works on Windows

**Use serial mode for development.** Only switch to parallel for production
runs on large note corpora (>10k notes).

Both backends produce identical output — you can verify with `.equals()`.

In [17]:
# Serial (default) — safest for development and debugging
out_serial = sud.extract_df(
    df=df,
    pattern_library=pattern_library,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    remove_linebreaks=True,
    parallel=False,  # serial
)

# Loky parallel — joblib-based, works on all platforms
out_loky = sud.extract_df(
    df=df,
    pattern_library=pattern_library,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    remove_linebreaks=True,
    parallel=True,
    parallel_backend="loky",
    n_workers=2,
)

# Outputs should be identical
print("Serial shape:", out_serial.shape)
print("Loky shape:  ", out_loky.shape)
print("Outputs identical:", out_serial.equals(out_loky))

Serial shape: (4, 71)
Loky shape:   (4, 71)
Outputs identical: True


## 9. File-based extraction — CLI

For large-scale production runs on full EHR corpora, use the CLI.
It chunks the input file automatically and writes to CSV.

### Standard CSV input

```bash
sudregex --extract \
  --in_file path/to/notes.csv \
  --out_file path/to/results.csv \
  --pattern-library path/to/my_pattern_library.py \
  --termslist path/to/termslist.py \
  --terms_active opioid_terms \
  --separator , \
  --person-column patient_id \
  --note-id-column note_id \
  --parallel \
  --parallel-backend loky \
  --n-workers 4
```

### Headerless file with custom separator (Vanderbilt SD DISCOVER format)

```bash
sudregex --extract \
  --in_file path/to/notes.txt \
  --out_file path/to/results.csv \
  --pattern-library path/to/my_pattern_library.py \
  --termslist path/to/termslist.py \
  --terms_active opioid_terms \
  --separator $'\t!\\^!\t' \
  --no-header \
  --columns patient_id,note_id,note_text \
  --parallel \
  --parallel-backend pandarallel \
  --n-workers 4
```

### Validate from the CLI

```bash
sudregex --validate \
  --pattern-library path/to/my_pattern_library.py \
  --examples path/to/validation_examples.txt \
  --val_out validation_detailed.csv \
  --val_by_item validation_by_item.csv \
  --print_mismatches
```

> **Note:** The `--checklist` flag is a deprecated alias for `--pattern-library`.
> It still works for backward compatibility but will be removed in a future version.

## 10. Bring your own pattern library

You can define a custom pattern library for any substance or clinical concept.
Each item is a dict with these required keys:

| Key | Type | Purpose |
|-----|------|---------|
| `lab` | str | Human-readable label for the item |
| `pat` | str or compiled regex | The regex pattern |
| `col_name` | str | Output column name |
| `substance` | bool | Require a substance term nearby? |
| `negation` | bool | Apply the negation gate? |
| `preview` | bool | Emit preview snippets for this item? |
| `common_fp` | list (optional) | Terms that indicate a false positive |

> **Tip:** `opioid=True` is a legacy alias for `substance=True`. Both work identically.

In [18]:
import re

# Example: a minimal custom pattern library for cocaine use
my_pattern_library = {
    "coc_active": {
        "lab": "Active cocaine use — self-report",
        "pat": re.compile(
            r"(patient|pt).{0,60}(report|admit|endors|disclos).{0,60}cocai"
            r"|cocai.{0,60}(per patient|per pt|self.?report)",
            re.IGNORECASE | re.MULTILINE,
        ),
        "col_name": "cocaine_active_use",
        "substance": True,   # require a cocaine term nearby
        "negation": True,    # apply negation gate
        "preview": True,     # emit snippets for QA
    },
    "coc_uds": {
        "lab": "UDS positive for cocaine",
        "pat": re.compile(
            r"(urine.drug.screen|uds|tox).{0,60}(positive|pos).{0,60}(cocaine|cocai|COC)"
            r"|(cocaine|COC).{0,40}(positive|detected)",
            re.IGNORECASE | re.MULTILINE,
        ),
        "col_name": "cocaine_uds_positive",
        "substance": False,  # substance name is in the pattern itself
        "negation": True,
        "preview": False,
        "common_fp": ["negative", "not detected", "COC negative"],
    },
}

# Define a matching termslist
cocaine_terms = [
    "cocaine", "cocai", "crack cocaine", "crack",
    "cocaine use disorder", "cocaine dependence",
    "COC", "benzoylecgonine",
]

# Run extraction with the custom pattern library
custom_notes = pd.DataFrame({
    "note_id":    ["C001", "C002", "C003"],
    "patient_id": ["Q001", "Q002", "Q003"],
    "note_text": [
        "Patient reports using cocaine daily for the past 6 months.",
        "UDS positive for cocaine and benzodiazepines today.",
        "Denies cocaine use. No illicit substances per patient report.",
    ],
})

custom_results = sud.extract_df(
    df=custom_notes,
    pattern_library=my_pattern_library,
    terms=cocaine_terms,           # pass terms directly instead of using termslist
    person_column="patient_id",
    id_column="note_id",
    remove_linebreaks=True,
    negation_scope="left",
    parallel=False,
)

custom_results

,patient_id,note_id,cocaine_active_use,cocaine_active_use_SUBSTANCE_MATCHED,cocaine_active_use_SUBSTANCE_MATCHED_NEG,cocaine_uds_positive,cocaine_uds_positive_NEG
0,Q001,C001,1,1,1,0,0
1,Q002,C002,0,0,0,1,1
2,Q003,C003,1,1,0,0,0


## 11. Load a pattern library from a file

For production use, keep your pattern library in a `.py` file and load it
via file path. The file must define a variable named `pattern_library`.

```python
# my_pattern_library.py
import re

pattern_library = {
    "item_A": {
        "lab": "...",
        "pat": re.compile(r"..."),
        "col_name": "...",
        "substance": True,
        "negation": True,
        "preview": False,
    },
    ...
}
```

Then pass the file path to `extract_df()`:

```python
result = sud.extract_df(
    df=df,
    pattern_library="path/to/my_pattern_library.py",
    terms=my_terms,
    ...
)
```

> **Backward compatibility:** If your file defines `checklist` instead of
> `pattern_library`, it will still work — the package accepts both variable names.

## 12. Termslist — using named term groups

The termslist is a dict of named term groups. You can use any group
with `terms_active=` or access the list directly via `termslist["group_name"]`.

This is useful when running the same pattern library against different
substance populations — just swap the term group.

In [19]:
# Option 1: pass via termslist + terms_active (recommended for file-based workflows)
result_via_termslist = sud.extract_df(
    df=df,
    pattern_library=pattern_library,
    termslist=termslist,
    terms_active="opioid_terms",  # name of the group to use
    id_column="note_id",
    remove_linebreaks=True,
    parallel=False,
)

# Option 2: pass terms directly as a list (convenient for notebook workflows)
result_via_terms = sud.extract_df(
    df=df,
    pattern_library=pattern_library,
    terms=termslist["opioid_terms"],  # pass the list directly
    id_column="note_id",
    remove_linebreaks=True,
    parallel=False,
)

# Both approaches produce identical output
print("Outputs identical:", result_via_termslist.equals(result_via_terms))

Outputs identical: True


## 13. Troubleshooting

**Pattern is firing too broadly (high FP)**
- Add terms to `common_fp` in the pattern item
- Tighten the regex window (reduce `.{0,N}`)
- Set `substance=True` to require a substance term nearby

**Pattern is not firing when it should (high FN)**
- Check `failure_reason` in `validate_pattern_library()` output
- `negated` → the negation gate is suppressing real hits; review the note context
- `needs_substance` → substance term not within the window; expand the window or add terms to your termslist
- `no_raw_hit` → the regex itself does not match; test the pattern directly with `re.findall()`

**ZeroDivisionError in validation**
- Fixed in v0.1.7. Items with no positive examples (`expected=1`) return `NaN` for precision/recall/F1 instead of crashing.

**Parallel backend issues**
- If `pandarallel` fails, switch to `parallel_backend="loky"` — it works on all platforms including Windows
- Python 3.13 users may see a `fork()` deprecation warning from `pandarallel` — this is a known upstream issue and does not affect correctness

**Output column names changed**
- As of v0.1.7, `checklist` is renamed to `pattern_library` throughout the API
- The old `checklist=` parameter still works as a deprecated alias
- Column names in the output are now more descriptive (e.g. `illicit_drug_use` instead of `illicit_drugs`)